# 05 - Baseline B4: HistGradientBoostingClassifier

B4 es el modelo principal. Este cuaderno contiene la busqueda, el ajuste final por escenario y los experimentos pendientes: LightGBM, ablaciones y H2.

**No hay interruptores.** Cada celda que entrena revisa primero SUS archivos en el dataset privado de Kaggle:

- si estan todos, los baja a `/content/resultados/` y termina sin entrenar (segundos);
- si falta alguno, entrena lo que falta (puede tardar horas) y lo publica en Kaggle al terminar.

Para entrenar usa CPU + RAM alta (8 nucleos, ~51 GiB); la primera celda los muestra. Lo entrenado se guarda primero en `/content/resultados/` y las funciones reanudan desde ahi si la celda se vuelve a ejecutar en el mismo servidor. Si el servidor se pierde antes de publicar, se pierde lo no publicado. La copia exacta del cuaderno original esta en `docx/03Evidence/05_entrenamiento_B4_independiente_original.ipynb`.

In [ ]:
# PREPARACION DEL CUADERNO
# PROPOSITO: dejar este servidor de Colab listo SIN depender de otro cuaderno.
# - Clona el repo (o lo actualiza) e instala las librerias fijadas en requirements.txt.
# - Muestra nucleos de CPU y RAM. Para entrenar se espera CPU + RAM alta (8 nucleos, ~51 GiB).
# - Configura Kaggle: token de los secretos de Colab, de secrets.txt o pedido a mano (oculto).
# - Baja a /content/datos solo las tablas que falten en este servidor.
# Los RESULTADOS (CSV, JSON, .joblib) NO se guardan en Drive ni en el repo: viven en el dataset
# privado de Kaggle. Cada celda que entrena revisa SUS archivos en Kaggle: si ya estan, los baja
# a /content/resultados y no entrena; si faltan, entrena y los publica.
# Despues del git pull se "olvidan" los modulos ptcg_wp ya cargados para usar el codigo nuevo.
# ESPERADO: la linea de nucleos/RAM, "ya existe" o "bajado" para las 21 tablas y
# "Tablas listas | caracteristicas: 62".
import json
import os
import subprocess
import sys
import time
from pathlib import Path

REPO = "Adriel23456/Applied-AI-Agent-Project"
BRANCH = "04MainAIModelTrainingAndDevelopment"
REPO_DIR = Path("/content/repo")
if not REPO_DIR.exists():
    !git clone -b {BRANCH} https://github.com/{REPO}.git /content/repo
else:
    !git -C /content/repo fetch origin {BRANCH}
    !git -C /content/repo checkout {BRANCH}
    !git -C /content/repo pull --ff-only origin {BRANCH}
%cd /content/repo
!pip install -q -r requirements.txt
if str(REPO_DIR / "src") not in sys.path:
    sys.path.insert(0, str(REPO_DIR / "src"))
for nombre in [m for m in list(sys.modules) if m == "ptcg_wp" or m.startswith("ptcg_wp.")]:
    del sys.modules[nombre]

# Nucleos y RAM de este servidor, y credenciales de Kaggle
from ptcg_wp.colab import recursos, setup_kaggle
RECURSOS = recursos()
setup_kaggle()

import numpy as np
import pandas as pd
import pyarrow.parquet as pq
from IPython.display import display
from ptcg_wp import store
from ptcg_wp.features import feature_columns
from ptcg_wp.ingest import load_days

from sklearn import __version__ as sklearn_version
assert sklearn_version == "1.7.2", (f"Se requiere scikit-learn 1.7.2; version cargada: {sklearn_version}. "
                                    "Reinicia el runtime y ejecuta esta celda de nuevo.")

# Carpetas de trabajo: tablas en /content/datos y resultados en /content/resultados
DATA_DIR = store.DATOS_DIR
RESULTADOS = store.RESULTADOS_DIR
DATASET = store.load_dataset_id()
store.fetch(DATASET, DATA_DIR, store.expected_files(load_days("configs/dias.yaml")))

# Las 62 caracteristicas del modelo principal (sin ids de carta ni columnas meta)
schema = pq.ParquetFile(sorted(DATA_DIR.glob("estados_*.parquet"))[0]).schema.names
FEATURES = feature_columns(schema)
assert len(FEATURES) == 62, len(FEATURES)
SEED = 2026
SCENARIOS = ("A", "B", "C")

# Trazabilidad que se guarda en cada registro.json nuevo
COMMIT = subprocess.check_output(["git", "-C", str(REPO_DIR), "rev-parse", "HEAD"], text=True).strip()
DIRTY = bool(subprocess.check_output(["git", "-C", str(REPO_DIR), "status", "--porcelain"], text=True).strip())
RUN_INFO = {"commit": COMMIT, "working_tree_dirty": DIRTY, "seed": SEED,
            "fecha_utc": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime()), "recursos": RECURSOS}
print("Tablas listas | caracteristicas:", len(FEATURES), "| commit:", COMMIT[:7],
      "| cambios locales:", DIRTY)

## Parte 1: busqueda de hiperparametros

Una busqueda TPE, 20 trials y tres pliegues agrupados por `game_id`, sobre las 11 285 partidas que son `train` en A, B y C a la vez. Optimiza log loss y registra AUC y Brier. Cada pliegue usa vigilancia agrupada, paciencia 30 y maximo 2000 iteraciones. El trial 20 fue la finalista. La busqueda original tardo unas 2 h 20 min con ocho nucleos.

In [ ]:
# B4 PARTE 1 - BUSQUEDA UNICA DE HIPERPARAMETROS
# PROPOSITO: obtener la finalista de B4.
# - Si b4_candidatas.csv, b4_cv_trials.csv y b4_optuna.sqlite3 estan en Kaggle: se bajan y NO se busca.
# - Si falta alguno: busqueda Optuna TPE con CV de 3 pliegues agrupada por partida, SOLO con
#   partidas que son train en A, B y C (ningun hiperparametro ve val, calib ni test). Guarda por
#   pliegue en /content/resultados/b4/busqueda: si se corta, volver a ejecutar reanuda.
# - NO cambiar los parametros: definen el estudio.
# ESPERADO: 11 285 partidas de busqueda; finalista trial 20 (AUC 0.838, log loss 0.488, Brier 0.163 en CV).
import faulthandler
import gc
from tempfile import TemporaryDirectory
from ptcg_wp.b4 import b4_optuna_cv, materialize_split
from ptcg_wp.baselines import ScenarioStream
from ptcg_wp.checkpoints import atomic_csv

B4_TRIALS = 20
B4_SPLITS = 3
B4_SEARCH_FRACTION = 1.0
B4_MAX_ITER = 2000
B4_PATIENCE = 30
B4_OBJECTIVE = "log_loss"
MAX_RAM_GIB = 10.0
BUSQUEDA_B4 = RESULTADOS / "b4" / "busqueda"
ESPERADOS_BUSQUEDA_B4 = ["b4/busqueda/b4_candidatas.csv", "b4/busqueda/b4_cv_trials.csv",
                         "b4/busqueda/b4_optuna.sqlite3"]


def entrenar_busqueda_b4():
    BUSQUEDA_B4.mkdir(parents=True, exist_ok=True)
    part = pd.read_parquet(DATA_DIR / "particiones.parquet")
    comunes = part.loc[(part["split_A"] == "train") & (part["split_B"] == "train")
                       & (part["split_C"] == "train"), "game_id"]
    print("partidas de busqueda (train en A, B y C):", len(comunes))
    trazas = open("/content/b4_pipeline_tracebacks.log", "a", buffering=1)
    faulthandler.enable(file=trazas)
    faulthandler.dump_traceback_later(600, repeat=True, file=trazas)
    try:
        with TemporaryDirectory(prefix="b4-busqueda-", dir="/content") as cache:
            stream = ScenarioStream(DATA_DIR, "A", tuple(FEATURES), batch_size=32768)
            train = materialize_split(stream, FEATURES, "train", cache)
            try:
                codigos = np.flatnonzero(train.games.isin(comunes))
                _, candidatas = b4_optuna_cv(
                    train, n_splits=B4_SPLITS, n_trials=B4_TRIALS, timeout=None, seed=SEED,
                    max_iter=B4_MAX_ITER, patience=B4_PATIENCE, max_ram_gib=MAX_RAM_GIB,
                    output_dir=BUSQUEDA_B4, study_cache_dir="/content/ptcg_optuna/b4_busqueda",
                    verbose=True, objective_metric=B4_OBJECTIVE,
                    search_fraction=B4_SEARCH_FRACTION, search_groups=codigos)
                atomic_csv(candidatas.to_dict("records"), BUSQUEDA_B4 / "b4_candidatas.csv")
            finally:
                train.close()
        gc.collect()
    finally:
        faulthandler.cancel_dump_traceback_later()
        faulthandler.disable()
        trazas.close()


store.asegurar(ESPERADOS_BUSQUEDA_B4, entrenar_busqueda_b4, "B4 busqueda de hiperparametros")
# Se leen siempre del CSV para que todas las sesiones usen exactamente las mismas finalistas
candidates = pd.read_csv(BUSQUEDA_B4 / "b4_candidatas.csv")
print(candidates.to_string(index=False))

## Parte 2: ajuste final por escenario

La finalista se ajusta con el `train` de cada escenario. El 15 % de las partidas de `train` decide el numero de arboles con parada temprana; despues se reajusta con todo `train`, con tope 4000. Los modelos usaron 2888, 2370 y 2714 iteraciones para A, B y C (unos 50 min por escenario con ocho nucleos). Cada escenario se publica en Kaggle apenas termina.

In [ ]:
# B4 PARTE 2 - AJUSTE FINAL CON TODO EL TRAIN DE CADA ESCENARIO (TOPE 4000)
# PROPOSITO: por escenario, bajar de Kaggle el modelo final si ya existe; si falta, reentrenar la
# finalista con todo su train, medirla en val y publicarla. test NO se toca.
# REQUISITO: haber ejecutado la parte 1 (define candidates).
# ESPERADO: A arboles=2888 AUC=0.8346 | B arboles=2370 AUC=0.7863 | C arboles=2714 AUC=0.8355.
from ptcg_wp.b4 import fit_b4_candidates_val

B4_MAX_ITER_FINAL = 4000
modelos_b4 = [f"b4_{fila.seleccionada_por}_CVtrial{fila.trial}.joblib" for fila in candidates.itertuples()]
for escenario in SCENARIOS:
    base = f"b4/final/escenario_{escenario}"
    esperados = [f"{base}/b4_val_finalistas.csv", f"{base}/b4_val_manifest.json"] + [
        f"{base}/{m}" for m in modelos_b4]
    print(f"--- Escenario {escenario} ---")

    def entrenar_final_b4(escenario=escenario, base=base):
        salida = RESULTADOS / base
        salida.mkdir(parents=True, exist_ok=True)
        with TemporaryDirectory(prefix=f"b4-final-{escenario}-", dir="/content") as cache:
            stream = ScenarioStream(DATA_DIR, escenario, tuple(FEATURES), batch_size=32768)
            train = materialize_split(stream, FEATURES, "train", cache)
            val = materialize_split(stream, FEATURES, "val", cache)
            try:
                metricas = fit_b4_candidates_val(
                    train, val, candidates, seed=SEED, max_iter=B4_MAX_ITER_FINAL,
                    patience=B4_PATIENCE, max_ram_gib=MAX_RAM_GIB, output_dir=salida,
                    scenario=escenario, verbose=True)
                atomic_csv(metricas.to_dict("records"), salida / "b4_val_finalistas.csv")
            finally:
                train.close()
                val.close()
        gc.collect()

    store.asegurar(esperados, entrenar_final_b4, f"B4 final tope 4000 escenario {escenario}")
    val = pd.read_csv(RESULTADOS / base / "b4_val_finalistas.csv")
    g = val.loc[val["etapa"].eq("global")].iloc[0]
    print(f"B4 {escenario} | arboles={g['n_iter_vigilancia']} | AUC={g['auc']:.4f} "
          f"log_loss={g['log_loss']:.4f} brier={g['brier']:.4f}")

## Configuracion elegida

HistGradientBoostingClassifier, trial 20: `learning_rate=0.06943`, `max_leaf_nodes=31`, `min_samples_leaf=50` y `l2_regularization=4.2503`. Cada escenario se ajusto con su propio `train` y se evaluo en su `val`; las metricas se reportan por escenario, no se promedian.

## Comparacion con LightGBM: protocolo definido

Presupuesto comparable con B4: TPE con 20 trials y 3 pliegues por partida, sobre la interseccion de partidas `train` de A/B/C. Optimiza log loss y registra AUC y Brier. Vigilancia agrupada del 15 %, paciencia 30, tope 2000 en busqueda y 4000 en el ajuste final; semilla 2026. Una sola finalista se elige por mayor AUC de CV (empates: menor log loss, menor Brier, numero de trial), antes de consultar `val`, y se ajusta con el `train` propio de cada escenario.

Se seleccionara LightGBM solo si, en cada escenario A/B/C, el intervalo percentil del 95 % de la diferencia de AUC (LightGBM - B4) queda por encima de cero y no aumentan los valores observados de log loss ni Brier (1000 remuestreos emparejados de partidas). Si no se cumple toda la regla se mantiene B4. Es una regla de seleccion en `val`, no la evaluacion final.

In [ ]:
# Version fijada de LightGBM (rapido si ya esta instalada)
%pip install -q lightgbm==4.6.0

In [ ]:
# LIGHTGBM: BUSQUEDA, AJUSTE FINAL Y COMPARACION CON B4
# PROPOSITO: tres bloques, cada uno revisa SUS archivos en Kaggle antes de entrenar.
#   1) busqueda (20 trials) -> lightgbm/*.csv, .json y .sqlite3
#   2) ajuste final por escenario -> lightgbm/escenario_X/
#   3) comparacion emparejada con B4 en val -> lightgbm/comparacion_b4/
# Si un bloque ya esta en Kaggle, se baja y no se repite. LGB_NUCLEOS queda fijo porque forma
# parte del manifiesto del estudio (cambiarlo impide reanudar).
# ESPERADO: "Decision sobre val: mantener_b4" o "seleccionar_lightgbm".
import lightgbm
from ptcg_wp.pending_experiments import compare_lightgbm_b4, run_lightgbm

assert lightgbm.__version__ == "4.6.0", "Instala la version fijada en la celda anterior."
LGB = RESULTADOS / "lightgbm"
LGB_NUCLEOS = 8
ESPERADOS_LGB_BUSQUEDA = [f"lightgbm/{n}" for n in (
    "registro.json", "lightgbm_config.json", "lightgbm_optuna.sqlite3", "lightgbm_cv_trials.csv",
    "lightgbm_cv_folds.csv", "lightgbm_finalista.csv")]


def entrenar_lightgbm():
    # run_lightgbm hace busqueda y ajustes finales; reanuda lo que ya este en LGB
    LGB.mkdir(parents=True, exist_ok=True)
    if not (LGB / "registro.json").exists():
        (LGB / "registro.json").write_text(json.dumps(RUN_INFO, indent=2))
    run_lightgbm(DATA_DIR, FEATURES, LGB, seed=SEED, n_jobs=LGB_NUCLEOS)


# 1) Busqueda
store.asegurar(ESPERADOS_LGB_BUSQUEDA, entrenar_lightgbm, "LightGBM busqueda")
trial_lgb = int(pd.read_csv(LGB / "lightgbm_finalista.csv").iloc[0]["trial"])
print("Finalista LightGBM: trial", trial_lgb)

# 2) Ajuste final por escenario (si la busqueda vino de Kaggle, run_lightgbm la reutiliza)
for escenario in SCENARIOS:
    print(f"--- LightGBM escenario {escenario} ---")
    store.asegurar([f"lightgbm/escenario_{escenario}/lightgbm_trial_{trial_lgb}.joblib",
                    f"lightgbm/escenario_{escenario}/lightgbm_val.csv"],
                   entrenar_lightgbm, f"LightGBM final escenario {escenario}")

# 3) Comparacion con B4 (necesita los modelos finales de B4)
ESPERADOS_LGB_COMPARACION = [f"lightgbm/comparacion_b4/{n}" for n in (
    "metricas_val.csv", "comparacion_val.csv", "decision_val.json",
    "bootstrap_auc_A.csv", "bootstrap_auc_B.csv", "bootstrap_auc_C.csv")]


def comparar_lightgbm():
    assert store.listo([f"b4/final/escenario_{e}/b4_auc_CVtrial20.joblib" for e in SCENARIOS])
    compare_lightgbm_b4(DATA_DIR, RESULTADOS / "b4" / "final", LGB, seed=SEED)


store.asegurar(ESPERADOS_LGB_COMPARACION, comparar_lightgbm, "LightGBM contra B4 en val")
display(pd.read_csv(LGB / "comparacion_b4" / "comparacion_val.csv").round(4))
print("Decision sobre val:", json.loads((LGB / "comparacion_b4" / "decision_val.json").read_text())["decision"])

## Ablaciones por familia

Tres variantes omiten una familia: progreso, recursos/tablero o diferenciales. La configuracion de B4 (trial 20) se mantiene fija; cada variante se ajusta con el `train` propio y se mide en `val` de cada escenario. Son nueve ajustes finales mas sus ajustes de vigilancia. `familias.json` registra las columnas exactas. Cada escenario se publica al terminar.

In [ ]:
# ABLACIONES DE B4 POR FAMILIA DE CARACTERISTICAS
# PROPOSITO: por escenario, bajar de Kaggle las tres variantes si ya existen; si falta alguna,
# ajustarlas (unas 2-3 h por escenario) y publicarlas.
# ESPERADO: por escenario, AUC val de sin_progreso, sin_recursos_tablero y sin_diferenciales.
from ptcg_wp.pending_experiments import feature_families, run_b4_ablation

assert store.listo(["b4/busqueda/b4_candidatas.csv"])
candidata_b4 = pd.read_csv(RESULTADOS / "b4/busqueda/b4_candidatas.csv")
candidata_b4 = candidata_b4.loc[candidata_b4["trial"].eq(20)].copy()
assert len(candidata_b4) == 1
fila = candidata_b4.iloc[0]
modelo_ablacion = f"b4_{fila['seleccionada_por']}_CVtrial{fila['trial']}.joblib"
familias = list(feature_families(FEATURES))
for escenario in SCENARIOS:
    base = f"b4/ablaciones/escenario_{escenario}"
    esperados = [f"{base}/familias.json", f"{base}/registro.json"] + [
        f"{base}/sin_{f}/{n}" for f in familias
        for n in ("b4_val_finalists.csv", "b4_val_manifest.json", "duracion.json", modelo_ablacion)]
    print(f"--- Ablaciones escenario {escenario} ---")

    def entrenar_ablaciones(escenario=escenario, base=base):
        salida = RESULTADOS / base
        salida.mkdir(parents=True, exist_ok=True)
        (salida / "registro.json").write_text(json.dumps(RUN_INFO, indent=2))
        run_b4_ablation(DATA_DIR, escenario, FEATURES, candidata_b4, salida, seed=SEED)

    store.asegurar(esperados, entrenar_ablaciones, f"B4 ablaciones escenario {escenario}")
    for f in familias:
        datos = pd.read_csv(RESULTADOS / base / f"sin_{f}" / "b4_val_finalists.csv")
        print("sin", f)
        display(datos.loc[datos["etapa"].eq("global")].round(4))

## H2: estado completo frente a cuatro entradas

Ajusta la variante restringida con turno, primer jugador y los dos ids activos, y la compara con el B4 completo ya guardado. Los ids se codifican como categorias con vocabulario de `train`; ids no vistos en `val` se tratan como ausentes. Si hay mas de 255 categorias, la celda se detiene para definir otra codificacion.

In [ ]:
# H2: VARIANTE RESTRINGIDA DE CUATRO ENTRADAS
# PROPOSITO: por escenario, bajar de Kaggle H2 si ya existe; si falta, ajustarlo y publicarlo.
# Muestra B4 completo y H2 restringido lado a lado (metricas globales de val).
from ptcg_wp.pending_experiments import run_h2

for escenario in SCENARIOS:
    base = f"b4/h2/escenario_{escenario}"
    print(f"--- H2 escenario {escenario} ---")

    def entrenar_h2(escenario=escenario, base=base):
        salida = RESULTADOS / base
        salida.mkdir(parents=True, exist_ok=True)
        (salida / "registro.json").write_text(json.dumps(RUN_INFO, indent=2))
        run_h2(DATA_DIR, escenario, salida, seed=SEED)

    store.asegurar([f"{base}/{n}" for n in ("h2_val.csv", "h2_restringido.joblib",
                                            "h2_config.json", "registro.json")],
                   entrenar_h2, f"H2 escenario {escenario}")
    completo_csv = f"b4/final/escenario_{escenario}/b4_val_finalistas.csv"
    assert store.listo([completo_csv])
    completo = pd.read_csv(RESULTADOS / completo_csv)
    restringido = pd.read_csv(RESULTADOS / base / "h2_val.csv")
    display(pd.concat([completo, restringido]).loc[lambda d: d["etapa"].eq("global")].round(4))

## Resumen

Metricas globales de `val` por escenario de todo lo que este disponible en este servidor: B4 completo, ablaciones, H2 y LightGBM.

In [ ]:
# RESUMEN DE METRICAS GLOBALES EN VAL (SOLO LECTURA)
# Muestra lo que ya este en /content/resultados (las celdas anteriores lo bajan o lo producen).
filas = []
fuentes = {"B4 completo": "b4/final/escenario_{e}/b4_val_finalistas.csv",
           "H2 restringido": "b4/h2/escenario_{e}/h2_val.csv",
           "LightGBM": "lightgbm/escenario_{e}/lightgbm_val.csv"}
for escenario in SCENARIOS:
    for nombre, patron in fuentes.items():
        ruta = RESULTADOS / patron.format(e=escenario)
        if ruta.exists():
            g = pd.read_csv(ruta).loc[lambda d: d["etapa"].eq("global")].iloc[0]
            filas.append({"escenario": escenario, "modelo": nombre, "auc": g["auc"],
                          "log_loss": g["log_loss"], "brier": g["brier"]})
    for ruta in sorted((RESULTADOS / f"b4/ablaciones/escenario_{escenario}").glob("sin_*/b4_val_finalists.csv")):
        g = pd.read_csv(ruta).loc[lambda d: d["etapa"].eq("global")].iloc[0]
        filas.append({"escenario": escenario, "modelo": f"B4 {ruta.parent.name}", "auc": g["auc"],
                      "log_loss": g["log_loss"], "brier": g["brier"]})
display(pd.DataFrame(filas).round(4))